In [1]:
import requests
import pandas as pd

API_KEY = "41TzyKAe6c526RYT3LY7wjQTMYzohdnKlZ9h1YmN"
station_id = 13

base_url = f"https://open-api.cmuccdc.org/api/dustboy/data30day/{station_id}"

params = {"apikey": API_KEY}

with requests.Session() as s:
    r = s.get(base_url, params=params, timeout=30)
    r.raise_for_status()

    # กันเคส server ส่ง HTML/ข้อความ error กลับมาแต่ status 200 (บางระบบเป็นแบบนั้น)
    ctype = (r.headers.get("Content-Type") or "").lower()
    if "json" not in ctype:
        raise ValueError(f"Expected JSON but got Content-Type={ctype}. Body head={r.text[:200]}")

    data = r.json()

print("Final URL (requests built):", r.url)  # เอาไว้ debug ตอนพัฒนา
print("Records:", len(data) if hasattr(data, "__len__") else type(data))

df0 = pd.DataFrame(data)

print("Top-level columns:", df0.columns.tolist())
print(df0.head(1).to_dict(orient="records")[0])

# --- แตก field 'value' ---
if "value" not in df0.columns:
    raise KeyError(f"No 'value' in response. Columns={df0.columns.tolist()}")

val = df0.loc[0, "value"]  # ส่วนมากมี 1 แถว

# 1) ถ้า value เป็น list -> มักเป็น time-series
if isinstance(val, list):
    df = pd.DataFrame(val)

# 2) ถ้า value เป็น dict -> อาจเป็น dict เดียว หรือมี field ย่อยที่เป็น list
elif isinstance(val, dict):
    # ถ้าใน dict มี list ซ่อนอยู่ เช่น val["data"] เป็น list
    list_key = next((k for k, v in val.items() if isinstance(v, list)), None)
    if list_key is not None:
        df = pd.DataFrame(val[list_key])
    else:
        df = pd.DataFrame([val])

else:
    raise TypeError(f"Unexpected type for value: {type(val)}")

print("Expanded columns:", df.columns.tolist())
print(df.head())

# --- หา time column แบบยืดหยุ่น ---
candidate_time_cols = ["datetime", "log_datetime", "time", "date_time", "timestamp", "date"]
time_col = next((c for c in candidate_time_cols if c in df.columns), None)
if time_col is None:
    raise KeyError(f"Cannot find time column in expanded value. Columns={df.columns.tolist()}")

df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

# --- หา pm25 column แบบยืดหยุ่น ---
candidate_pm25_cols = ["pm25", "PM25", "pm2_5", "pm2.5"]
pm25_col = next((c for c in candidate_pm25_cols if c in df.columns), None)
if pm25_col is None:
    raise KeyError(f"Cannot find pm25 column in expanded value. Columns={df.columns.tolist()}")

df[pm25_col] = pd.to_numeric(df[pm25_col], errors="coerce")
df = df.dropna(subset=[time_col, pm25_col])

# --- daily mean ---
daily_pm25 = (
    df.set_index(time_col)
      .resample("D")[pm25_col].mean()
      .reset_index()
      .rename(columns={time_col: "date", pm25_col: "pm25_daily_mean"})
)

print(daily_pm25.head())
print("จำนวนวัน:", len(daily_pm25))

Final URL (requests built): https://open-api.cmuccdc.org/api/dustboy/data30day/13?apikey=41TzyKAe6c526RYT3LY7wjQTMYzohdnKlZ9h1YmN
Records: 8
Top-level columns: ['id', 'dustboy_id', 'dustboy_name', 'dustboy_lat', 'dustboy_lon', 'version', 'model', 'value']
{'id': '13', 'dustboy_id': 'DustBoy H018', 'dustboy_name': 'มช. แคมปัสแม่เหียะ ต.สุเทพ อ.เมือง จ.เชียงใหม่', 'dustboy_lat': '18.761371', 'dustboy_lon': '98.931855', 'version': '', 'model': 'PRO', 'value': {'pm10': 56, 'pm25': 46, 'temp': 0, 'humid': 0, 'log_datetime': '2026-04-30 15:00'}}
Expanded columns: ['pm10', 'pm25', 'temp', 'humid', 'log_datetime']
   pm10  pm25  temp  humid      log_datetime
0    56    46     0      0  2026-04-30 15:00
        date  pm25_daily_mean
0 2026-04-30             46.0
จำนวนวัน: 1


In [2]:
import requests
import pandas as pd

API_KEY = "41TzyKAe6c526RYT3LY7wjQTMYzohdnKlZ9h1YmN"
station_id = 13

url = f"https://open-api.cmuccdc.org/api/dustboy/data30day/{station_id}"
r = requests.get(url, params={"apikey": API_KEY}, timeout=30)
r.raise_for_status()
data = r.json()

# ---------- 1) แปลง response ให้เป็นตาราง "รายชั่วโมง" ----------
# กรณี A: top-level เป็น list รายชั่วโมงเลย (มี datetime/log_datetime + pm25)
if isinstance(data, list) and data and isinstance(data[0], dict) and ("pm25" in data[0]):
    hourly = pd.DataFrame(data)

# กรณี B: top-level เป็น list ของ station แล้วข้อมูลรายชั่วโมงอยู่ใน value (มักเป็น list)
else:
    df0 = pd.DataFrame(data if isinstance(data, list) else [data])

    # เลือก record ของสถานีที่ต้องการ (เผื่อส่งมาหลายสถานี)
    rec = df0[df0["id"].astype(str) == str(station_id)]
    if rec.empty:
        rec = df0.iloc[[0]]
    val = rec.iloc[0].get("value")

    # value ต้องเป็น list ของรายชั่วโมง
    if isinstance(val, list):
        hourly = pd.DataFrame(val)
    else:
        raise ValueError("response นี้ไม่ได้มี time-series รายชั่วโมงใน 'value' (ได้แค่ snapshot เดียว)")

# ---------- 2) เลือกคอลัมน์เวลา ----------
time_col = None
for c in ["datetime", "log_datetime", "time", "timestamp", "date_time"]:
    if c in hourly.columns:
        time_col = c
        break
if time_col is None:
    raise KeyError(f"ไม่เจอคอลัมน์เวลาในข้อมูลรายชั่วโมง: {hourly.columns.tolist()}")

hourly[time_col] = pd.to_datetime(hourly[time_col], errors="coerce")
hourly["pm25"] = pd.to_numeric(hourly["pm25"], errors="coerce")
hourly = hourly.dropna(subset=[time_col, "pm25"])

# ---------- 3) รายวัน: เฉลี่ย pm25 ----------
daily = (
    hourly.set_index(time_col)
          .resample("D")["pm25"]
          .mean()
          .reset_index()
          .rename(columns={time_col: "date", "pm25": "pm25_daily_mean"})
)

print(daily.head())
print("จำนวนวัน:", len(daily))


        date  pm25_daily_mean
0 2026-03-31       122.833333
1 2026-04-01       154.708333
2 2026-04-02       154.625000
3 2026-04-03       158.833333
4 2026-04-04       213.250000
จำนวนวัน: 31


In [1]:
import requests
import pandas as pd

API_KEY = "41TzyKAe6c526RYT3LY7wjQTMYzohdnKlZ9h1YmN"

url = "https://open-api.cmuccdc.org/api/dustboy/geography"
r = requests.get(url, params={"apikey": API_KEY}, timeout=30)
r.raise_for_status()

data = r.json()
df = pd.DataFrame(data)

stations = df[["id", "dustboy_name", "province_id","dustboy_lat", "dustboy_lon"]].rename(
    columns={"id": "station_id"}
)

print(stations.head())
print("จำนวนสถานีทั้งหมด:", len(stations))

  station_id                                    dustboy_name province_id  \
0          4    ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่          38   
1         12   Bear Hug Cafe‚ ต.ช้างคลาน อ.เมือง จ.เชียงใหม่          38   
2         13  มช. แคมปัสแม่เหียะ ต.สุเทพ อ.เมือง จ.เชียงใหม่          38   
3         14                รพ.ฝาง ต.เวียง อ.ฝาง จ.เชียงใหม่          38   
4         16  โรงพยาบาลร้องกวาง ต.ร้องเข็ม อ.ร้องกวาง จ.แพร่          42   

          dustboy_lat         dustboy_lon  
0          18.7854765          98.9996602  
1          18.7828996          98.9933469  
2           18.761371           98.931855  
3           19.913293           99.206827  
4  18.307634262860546  100.29124223068229  
จำนวนสถานีทั้งหมด: 261


In [12]:
yyyy = 2026
mm = "04"
apikey = "41TzyKAe6c526RYT3LY7wjQTMYzohdnKlZ9h1YmN"
url = f"https://open-api.cmuccdc.org/api/dustboy/database/{yyyy}{mm}"
r = requests.get(url, params={"apikey": API_KEY})
r.raise_for_status()

In [16]:
data = r.json()
with open(f'database_{yyyy}{mm}.json', 'w') as f:
    f.write(r.text)

In [4]:
import pandas as pd
import re

def extract_between_markers(text: str, marker: str, next_markers=("ต.", "อ.", "จ.")):
    """
    ดึงข้อความหลัง marker (เช่น 'อ.') ไปจนถึงก่อน marker ถัดไป (ต./อ./จ.) หรือจบข้อความ
    รองรับช่องว่างหลังจุด เช่น 'จ. เชียงใหม่'
    """
    if not isinstance(text, str):
        return None

    # สร้าง lookahead สำหรับ marker ถัดไป
    next_pat = "|".join(re.escape(m) for m in next_markers)

    # marker อนุญาตให้มี/ไม่มีช่องว่างหลังจุด
    pat = rf"{re.escape(marker)}\s*(.+?)(?=\s*(?:{next_pat})\s*|$)"

    m = re.search(pat, text)
    if not m:
        return None

    # เก็บข้อความที่จับได้ แล้ว trim เฉพาะหัวท้าย (ไม่แตะช่องว่างข้างใน)
    value = m.group(1).strip()

    # กันเครื่องหมายคั่นแปลก ๆ ที่ติดท้าย เช่น ',' '‚' ';'
    value = re.sub(r"[,\u201a;]+$", "", value).strip()

    return value if value else None


def split_th_address_v2(name: str):
    tambon = extract_between_markers(name, "ต.")
    amphoe = extract_between_markers(name, "อ.")
    province = extract_between_markers(name, "จ.")
    return pd.Series([tambon, amphoe, province])

# df คือ DataFrame ที่มี dustboy_name
stations[["tambon", "amphoe", "province"]] = stations["dustboy_name"].apply(split_th_address_v2)

print(stations[["dustboy_name", "tambon", "amphoe", "province","dustboy_lat", "dustboy_lon"]].head())


                                     dustboy_name    tambon    amphoe  \
0    ไนท์บาร์ซาร์ ต.ช้างม่อย อ.เมือง จ. เชียงใหม่  ช้างม่อย     เมือง   
1   Bear Hug Cafe‚ ต.ช้างคลาน อ.เมือง จ.เชียงใหม่  ช้างคลาน     เมือง   
2  มช. แคมปัสแม่เหียะ ต.สุเทพ อ.เมือง จ.เชียงใหม่     สุเทพ     เมือง   
3                รพ.ฝาง ต.เวียง อ.ฝาง จ.เชียงใหม่     เวียง       ฝาง   
4  โรงพยาบาลร้องกวาง ต.ร้องเข็ม อ.ร้องกวาง จ.แพร่  ร้องเข็ม  ร้องกวาง   

    province         dustboy_lat         dustboy_lon  
0  เชียงใหม่          18.7854765          98.9996602  
1  เชียงใหม่          18.7828996          98.9933469  
2  เชียงใหม่           18.761371           98.931855  
3  เชียงใหม่           19.913293           99.206827  
4       แพร่  18.307634262860546  100.29124223068229  


In [5]:
print(stations[["station_id", "tambon", "amphoe", "province","dustboy_lat", "dustboy_lon"]])

    station_id    tambon    amphoe   province         dustboy_lat  \
0            4  ช้างม่อย     เมือง  เชียงใหม่          18.7854765   
1           12  ช้างคลาน     เมือง  เชียงใหม่          18.7828996   
2           13     สุเทพ     เมือง  เชียงใหม่           18.761371   
3           14     เวียง       ฝาง  เชียงใหม่           19.913293   
4           16  ร้องเข็ม  ร้องกวาง       แพร่  18.307634262860546   
..         ...       ...       ...        ...                 ...   
250       7126  เชียงดาว  เชียงดาว  เชียงใหม่           19.413831   
251       7136   สันทราย     พร้าว  เชียงใหม่           19.416707   
252       7156   แม่ปั๋ง     พร้าว  เชียงใหม่            19.22102   
253       7166  จางเหนือ   แม่เมาะ      ลำปาง           18.491297   
254       7171  จางเหนือ   แม่เมาะ      ลำปาง           18.357949   

            dustboy_lon  
0            98.9996602  
1            98.9933469  
2             98.931855  
3             99.206827  
4    100.29124223068229  
..             

In [6]:
station_df = stations[["station_id", "dustboy_name", "tambon", "amphoe", "province","dustboy_lat", "dustboy_lon"]]

station_df.to_csv(
    "station_df.csv",
    index=False,      # ไม่บันทึก index
    encoding="utf-8-sig"  # เหมาะกับภาษาไทย เปิดใน Excel ได้สวย
)


# k-NN + rule-based (non-province)

In [7]:
import re
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# =========================
# 0) Helpers
# =========================
def norm_th(s):
    """Normalize แบบไม่ทำให้ข้อความเพี้ยน: ไม่ลบช่องว่างทั้งหมด"""
    if pd.isna(s):
        return s
    s = str(s).replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"(ต\.|อ\.|จ\.)\s+", r"\1", s)  # "จ. เชียงใหม่" -> "จ.เชียงใหม่"
    return s

def pick_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for k in candidates:
        if k.lower() in cols:
            return cols[k.lower()]
    return None

def fit_balltree(lat, lon):
    X = np.deg2rad(np.c_[lat, lon])
    return BallTree(X, metric="haversine")

def query_knn(tree, q_lat, q_lon, k=30):
    q = np.deg2rad(np.c_[q_lat, q_lon])
    dist, ind = tree.query(q, k=k)
    dist_km = dist * 6371.0
    return ind, dist_km

def strip_province_prefix(p):
    # รองรับ NaN / ตัวเลข / object แปลก ๆ
    if pd.isna(p):
        return ""
    p = norm_th(str(p))
    p = re.sub(r"^จ\.\s*", "", p).strip()
    return p

def strip_changwat(p):
    if pd.isna(p):
        return ""
    p = norm_th(str(p))
    return re.sub(r"^จ\.\s*", "", p).strip()

def extract_province_hint_from_text(text, provinces_set):
    """
    ถ้าใน dustboy_name มีชื่อจังหวัดแบบเต็ม ๆ ให้คืนชื่อจังหวัดนั้น
    ใช้ provinces_set จาก ref เพื่อไม่ต้อง hardcode รายชื่อ 77 จังหวัด
    """
    t = norm_th(text)
    # เช็คแบบ contains ชื่อจังหวัด
    for prov in provinces_set:
        if prov and prov in t:
            return prov
    return None

In [8]:
# =========================
# 1) Load
# =========================
stations = pd.read_csv("station_df.csv")
ref = pd.read_csv("thai_tambon_key.csv")

# normalize
stations["dustboy_name"] = stations.get("dustboy_name", "").apply(norm_th)
if "province" in stations.columns:
    stations["province"] = stations["province"].apply(strip_province_prefix)

# ref columns (ของคุณ)
ref["TAMBON_T"] = ref["TAMBON_T"].apply(lambda x: norm_th("" if pd.isna(x) else str(x)))
ref["AMPHOE_T"] = ref["AMPHOE_T"].apply(lambda x: norm_th("" if pd.isna(x) else str(x)))
ref["CHANGWAT_T"] = ref["CHANGWAT_T"].apply(strip_changwat)


# lat/lon
st_lat = pick_col(stations, ["dustboy_lat", "lat", "latitude"])
st_lon = pick_col(stations, ["dustboy_lon", "lon", "lng", "longitude"])
ref_lat = pick_col(ref, ["LAT", "lat", "latitude"])
ref_lon = pick_col(ref, ["LONG", "lon", "lng", "longitude"])

if st_lat is None or st_lon is None:
    raise ValueError(f"station_df.csv ไม่พบ lat/lon: {list(stations.columns)}")
if ref_lat is None or ref_lon is None:
    raise ValueError(f"thai_tambon_key.csv ไม่พบ LAT/LONG: {list(ref.columns)}")

stations[st_lat] = pd.to_numeric(stations[st_lat], errors="coerce")
stations[st_lon] = pd.to_numeric(stations[st_lon], errors="coerce")
ref[ref_lat] = pd.to_numeric(ref[ref_lat], errors="coerce")
ref[ref_lon] = pd.to_numeric(ref[ref_lon], errors="coerce")

stations_valid = stations.dropna(subset=[st_lat, st_lon]).copy()
ref_valid = ref.dropna(subset=[ref_lat, ref_lon]).copy()

# รายชื่อจังหวัดจาก ref (77 จังหวัด)
provinces_set = set(ref_valid["CHANGWAT_T"].dropna().unique().tolist())

# province hint: ใช้ province column ถ้ามี, ถ้าไม่มีค่อยสกัดจากชื่อสถานี
prov_hint = []
for _, r in stations_valid.iterrows():
    p = None
    if "province" in stations_valid.columns and isinstance(r.get("province", ""), str) and r["province"].strip():
        p = r["province"].strip()
    if not p:
        p = extract_province_hint_from_text(r["dustboy_name"], provinces_set)
    prov_hint.append(p)

stations_valid["province_hint"] = prov_hint  # อาจเป็น None ได้

FileNotFoundError: [Errno 2] No such file or directory: 'thai_tambon_key.csv'

In [ ]:
# =========================
# 2) k-NN candidates
# =========================
K = 20  # ปรับได้: 20-50
tree = fit_balltree(ref_valid[ref_lat].values, ref_valid[ref_lon].values)
inds, dists_km = query_knn(tree, stations_valid[st_lat].values, stations_valid[st_lon].values, k=K)

# =========================
# 3) Pick best candidate with province hint (optional)
# =========================
ref_prov = ref_valid["CHANGWAT_T"].values
ref_tam  = ref_valid["TAMBON_T"].values
ref_amp  = ref_valid["AMPHOE_T"].values

best_idx = []
best_dist = []
picked_reason = []

for i in range(len(stations_valid)):
    cand_idx = inds[i]
    cand_dist = dists_km[i]
    hint = stations_valid.iloc[i]["province_hint"]

    if hint:  # ถ้ามี hint จังหวัด -> เลือก candidate ที่จังหวัดตรงและระยะใกล้สุด
        mask = (ref_prov[cand_idx] == hint)
        if mask.any():
            j = np.argmin(cand_dist[mask])
            chosen = cand_idx[np.where(mask)[0][j]]
            dist = cand_dist[mask][j]
            reason = "prov_hint_match"
        else:
            # ถ้าไม่มีจังหวัดตรงใน top-K -> เลือกใกล้สุดไปก่อน แต่ทำ flag ไว้
            chosen = cand_idx[np.argmin(cand_dist)]
            dist = cand_dist.min()
            reason = "nearest_no_prov_match"
    else:
        # ไม่มี hint -> ใกล้สุดล้วน ๆ
        chosen = cand_idx[np.argmin(cand_dist)]
        dist = cand_dist.min()
        reason = "nearest_no_hint"

    best_idx.append(chosen)
    best_dist.append(dist)
    picked_reason.append(reason)

stations_valid["tambon_geo"] = ref_tam[best_idx]
stations_valid["amphoe_geo"] = ref_amp[best_idx]
stations_valid["province_geo"] = ref_prov[best_idx]
stations_valid["match_dist_km"] = best_dist
stations_valid["pick_reason"] = picked_reason

In [ ]:
# =========================
# 4) "เชื่อ geo เป็นหลัก" + เก็บ orig ไว้ตรวจ
# =========================
stations_valid["tambon_orig"] = stations_valid.get("tambon", np.nan)
stations_valid["amphoe_orig"] = stations_valid.get("amphoe", np.nan)
stations_valid["province_orig"] = stations_valid.get("province", np.nan)

stations_valid["tambon"] = stations_valid["tambon_geo"].combine_first(stations_valid["tambon_orig"])
stations_valid["amphoe"] = stations_valid["amphoe_geo"].combine_first(stations_valid["amphoe_orig"])
stations_valid["province"] = stations_valid["province_geo"].combine_first(stations_valid["province_orig"])

stations_valid["admin_complete"] = stations_valid[["tambon","amphoe","province"]].notna().all(axis=1)
stations_valid["admin_changed"] = (
    (stations_valid["tambon_orig"] != stations_valid["tambon"]) |
    (stations_valid["amphoe_orig"] != stations_valid["amphoe"]) |
    (stations_valid["province_orig"] != stations_valid["province"])
)

# quality label
stations_valid["match_quality"] = np.select(
    [stations_valid["match_dist_km"] <= 5,
     stations_valid["match_dist_km"] <= 15],
    ["good", "ok"],
    default="suspect"
)

In [ ]:
# =========================
# 5) Merge back + Save (เฉพาะคอลัมน์ที่ต้องการ)
# =========================

# 5.1 สร้างตารางสำหรับ merge โดยตั้งชื่อคอลัมน์ใหม่เป็น *_new
merge_df = stations_valid[[
    "station_id",
    "tambon_geo", "amphoe_geo", "province_geo",
    "match_dist_km", "match_quality", "pick_reason",
    "admin_complete", "admin_changed"
]].copy()

merge_df = merge_df.rename(columns={
    "tambon_geo": "tambon_new",
    "amphoe_geo": "amphoe_new",
    "province_geo": "province_new"
})

# 5.2 merge กลับเข้า stations (ไม่ต้อง suffix แล้ว)
stations_out = stations.merge(merge_df, on="station_id", how="left")

# 5.3 เลือกเฉพาะคอลัมน์ตามที่คุณต้องการ
final_cols = [
    "station_id",
    "dustboy_name",
    "dustboy_lat",
    "dustboy_lon",
    "tambon_new",
    "amphoe_new",
    "province_new",
    "match_dist_km",
    "match_quality",
    "pick_reason",
    "admin_complete",
    "admin_changed",
]
final_cols = [c for c in final_cols if c in stations_out.columns]
final_df = stations_out[final_cols].copy()

# 5.4 save ไฟล์หลัก
final_df.to_csv("station_enriched_admin.csv", index=False, encoding="utf-8-sig")

# 5.5 แยก suspect ไฟล์ QC
sus = final_df[final_df["match_quality"] == "suspect"].copy()
sus.to_csv("station_suspect_far_matches.csv", index=False, encoding="utf-8-sig")

# 5.6 report
print("✅ Saved: station_enriched_admin.csv")
print("✅ Saved: station_suspect_far_matches.csv")
print("Rows:", len(final_df))
print("Complete rate:", final_df["admin_complete"].mean())
print("Changed rate:", final_df["admin_changed"].mean())

print("\nTop-10 far matches:")
print(
    final_df.sort_values("match_dist_km", ascending=False)
    [["station_id","dustboy_name","province_new","match_dist_km","match_quality","pick_reason"]]
    .head(10)
)

✅ Saved: station_enriched_admin.csv
✅ Saved: station_suspect_far_matches.csv
Rows: 228
Complete rate: 1.0
Changed rate: 1.0

Top-10 far matches:
     station_id                                       dustboy_name  \
154        5619          สสอ.ขุนยวม ต.ขุนยวม อ.ขุนยวม จ.แม่ฮ่องสอน   
177        6135                             สสอ.อมก๋อย จ.เชียงใหม่   
134        5327      โรงพยาบาลอมก๋อย ต.อมก๋อย อ.อมก๋อย จ.เชียงใหม่   
132        5317           รพ.ขุนยวม ต.ขุนยวม อ.ขุนยวม จ.แม่ฮ่องสอน   
13           84        ทต.เมืองนะ ต.เมืองนะ อ.เชียงดาว จ.เชียงใหม่   
164        5698       รพ.ปางมะผ้า ต.สบป่อง อ.ปางมะผ้า จ.แม่ฮ่องสอน   
169        5713          รพ.สต.บ้านอรุโณทัย อ.เชียงดาว จ.เชียงใหม่   
11           71              รพ.นาหมื่น ต.บ่อแก้ว อ.นาหมื่น จ.น่าน   
192        6711  โรงเรียนบ้านดอนมหาวัน ต.เวียง อ.เชียงของ จ.เชี...   
168        5712                   รพ.ฮอด ต.หางดง อ.ฮอด จ.เชียงใหม่   

    province_new  match_dist_km match_quality      pick_reason  
154   แม่ฮ่องสอน   